In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
import wandb

# SETUP & CONFIGURATION

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(seed=42)

# Global constants
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Initialize WandB
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

# METRIC EVALUATION FUNCTION

In [5]:
def calculate_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argsort(logits, axis=-1)[:, ::-1] # Sort in descending order
    
    # Top-1 metrics
    top1_preds = preds[:, 0]
    accuracy = accuracy_score(labels, top1_preds)
    f1 = f1_score(labels, top1_preds, average="macro")

    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
    }

## HUGGING FACE TRANSFORMER DATASETS

In [7]:
class HuggingFaceMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        choices_inputs = []
        for opt in self.options:
            option_text = str(row[opt])
            choices_inputs.append((prompt, option_text))
            
        # Standard tokenization structure matching [Batch_Size, Num_Choices, Max_Length]
        features = self.tokenizer(
            [text[0] for text in choices_inputs],
            [text[1] for text in choices_inputs],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": features["input_ids"],
            "attention_mask": features["attention_mask"]
        }
        
        if not self.is_test:
            item["labels"] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
        return item

# Data collator to enforce precise shapes for Hugging Face multi-choice pipeline execution
def mcq_data_collator(features):
    batch = {}
    batch["input_ids"] = torch.stack([f["input_ids"] for f in features])
    batch["attention_mask"] = torch.stack([f["attention_mask"] for f in features])
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch

### MODEL: PRETRAINED DeBERTa

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Stratified validation splits
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])

deberta_ckpt = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_ckpt)
deberta_model = AutoModelForMultipleChoice.from_pretrained(deberta_ckpt).to(DEVICE)
deberta_train_ds = HuggingFaceMCQDataset(train_split, deberta_tokenizer, max_len=128)
deberta_val_ds = HuggingFaceMCQDataset(val_split, deberta_tokenizer, max_len=128)

NUM_EPOCHS = 3  

deberta_args = TrainingArguments(
    output_dir="./deberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,                
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                   
    max_grad_norm=0.5,
    per_device_train_batch_size=8,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   
    num_train_epochs=NUM_EPOCHS,
    load_best_model_at_end=True,
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name="pretrained-deberta-run",
    logging_steps=10,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)
deberta_trainer.train()
wandb.finish()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,4.694531,2.740234,0.235000,0.231908
2,4.430273,2.613281,0.435000,0.438983
3,5.198730,2.353516,0.510000,0.511220


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,▂▁▃▇█
eval/f1_score,▃▁▄▇█
eval/loss,██▄▃▁
eval/runtime,▃▁█▅▅
eval/samples_per_second,▆█▁▄▄
eval/steps_per_second,▆█▁▄▄
train/epoch,▁▂▂▃▃▃▄▄▅▅▅▅▁▂▂▃▃▃▄▄▅▅▅▅▆▇▇▇███
train/global_step,▁▁▂▃▃▃▃▄▅▅▅▅▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇███
train/grad_norm,▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▁█▄▃▁▁▁
train/learning_rate,▅██▇▇▆▆▅▄▃▅██▇▇▆▆▅▄▃▂▂▁▁▁
+1,...


In [ ]:
# PREDICTIONS FOR MODEL (DeBERTa)
# 1. Create the test dataset and dataloader for DeBERTa
test_deberta_ds = HuggingFaceMCQDataset(test_df, deberta_tokenizer, max_len=128, is_test=True)
deberta_test_loader = DataLoader(test_deberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# 2. Set the model to evaluation mode
deberta_model.eval()
deberta_probs = []

# 3. Run inference without calculating gradients
with torch.no_grad():
    for batch in deberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = deberta_model(**inputs).logits
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        deberta_probs.append(probs.cpu().numpy())
        
# Concatenate all batches into a single numpy array
deberta_probs = np.concatenate(deberta_probs, axis=0)

# 4. Extract Top-3 space-separated string maps for output submissions
deberta_submission_predictions = []
for probs in deberta_probs:
    # Sort indices in descending order based on probability and grab the top 3
    top3_indices = np.argsort(probs)[::-1][:3]
    # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    deberta_submission_predictions.append(" ".join(top3_labels))
    
# 5. Generate final Kaggle submission tracking file
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": deberta_submission_predictions
})

submission_df.to_csv("submission.csv", index=False)
print("Inference completed successfully. Output saved to: submission.csv")

# submission_df.info()
# submission_df.describe(include='all')

Inference completed successfully. Output saved to: submission.csv


.info() prints a concise summary of the DataFrame, including the index dtype and columns, non-null values, and memory usage.

.describe(include='all') generates descriptive statistics. Using include='all' ensures it also summarizes object/string columns showing counts, unique values, and the most frequent value.